# Combined playback-test localizations (Patches A and E)

This notebook combines the saved localization results from:

- `20260722__audiomoth_GPS_TESTatA7_localization_patchA_TEST1.ipynb`
- `20260722__audiomoth_GPS_TESTatA7_localization_patchA_TEST2.ipynb`
- `20260722__audiomoth_GPS_TESTatA7_localization_patchE_TEST3.ipynb`
- `20260722__audiomoth_GPS_TESTatA7_localization_patchE_TEST4.ipynb`

The first two columns of each localization result are east and north offsets in meters from that test's reference microphone. The third fitted value is retained in the data but is not used on the 2D Folium map.

In [37]:
from pathlib import Path

import numpy as np
import folium
from folium.plugins import MeasureControl

## Microphone locations

In [38]:
# Patch A ground-truth microphone locations (latitude, longitude).
patch_a_microphones = {
    "A1": [47 + (39.3437 / 60), -(122 + (17.8028 / 60))],
    "A4": [47 + (39.3243 / 60), -(122 + (17.8002 / 60))],
    "A7": [47 + (39.3061 / 60), -(122 + (17.8012 / 60))],
}

# Patch E average post locations (latitude, longitude), copied from the
# Patch E test notebooks.
patch_e_microphones = {
    "E2": [47.65483889, -122.29553889],
    "E4": [47.65494444, -122.29488333],
    "E8": [47.65444167, -122.29487500],
    "E9": [47.65428611, -122.29529444],
}

microphone_locations = {
    **{f"Patch A — {name}": loc for name, loc in patch_a_microphones.items()},
    **{f"Patch E — {name}": loc for name, loc in patch_e_microphones.items()},
}

microphone_locations

{'Patch A — A1': [47.655728333333336, -122.29671333333333],
 'Patch A — A4': [47.655405, -122.29667],
 'Patch A — A7': [47.65510166666667, -122.29668666666667],
 'Patch E — E2': [47.65483889, -122.29553889],
 'Patch E — E4': [47.65494444, -122.29488333],
 'Patch E — E8': [47.65444167, -122.294875],
 'Patch E — E9': [47.65428611, -122.29529444]}

## Saved localization results

In [39]:
notebook_dir = Path.cwd() / "daily_notebooks"
if not notebook_dir.is_dir():
    notebook_dir = Path.cwd()
localization_results_dir = notebook_dir / "localization_results"

localization_result_paths = {
    "patch_a_test1": localization_results_dir / "20260723_patchA_TEST1_all_playbacks_source_locs.npy",
    "patch_a_test2": localization_results_dir / "20260723_patchA_TEST2_all_playbacks_source_locs.npy",
    "patch_e_test3": localization_results_dir / "20260723_patchE_TEST3_all_playbacks_source_locs.npy",
    "patch_e_test4": localization_results_dir / "20260723_patchE_TEST4_all_playbacks_source_locs.npy",
}

missing_results = [path for path in localization_result_paths.values() if not path.exists()]
if missing_results:
    missing_list = "\n".join(f"- {path}" for path in missing_results)
    raise FileNotFoundError(
        "Missing saved localization arrays. Run the corresponding TEST notebooks first:\n"
        f"{missing_list}"
    )

patch_a_test1_all_playbacks_source_locs = np.load(
    localization_result_paths["patch_a_test1"], allow_pickle=False
)
patch_a_test2_all_playbacks_source_locs = np.load(
    localization_result_paths["patch_a_test2"], allow_pickle=False
)
patch_e_test3_all_playbacks_source_locs = np.load(
    localization_result_paths["patch_e_test3"], allow_pickle=False
)
patch_e_test4_all_playbacks_source_locs = np.load(
    localization_result_paths["patch_e_test4"], allow_pickle=False
)

loaded_source_locs = {
    "Patch A TEST1": (patch_a_test1_all_playbacks_source_locs, 5),
    "Patch A TEST2": (patch_a_test2_all_playbacks_source_locs, 5),
    "Patch E TEST3": (patch_e_test3_all_playbacks_source_locs, 10),
    "Patch E TEST4": (patch_e_test4_all_playbacks_source_locs, 6),
}
for result_name, (source_locs, expected_rows) in loaded_source_locs.items():
    expected_shape = (expected_rows, 3)
    if source_locs.shape != expected_shape:
        raise ValueError(
            f"{result_name} has shape {source_locs.shape}; expected {expected_shape}. "
            "Rerun and save the corresponding source notebook."
        )

playback_tests = {
    1: {
        "patch": "A",
        "color": "red",
        "reference_name": "A7",
        "reference_latlon": patch_a_microphones["A7"],
        "times": ["12:02:57", "12:03:27", "12:03:57", "12:04:27", "12:04:57"],
        "source_files": ["*_190000_SYNC.WAV"] * 5,
        "source_locs": patch_a_test1_all_playbacks_source_locs,
    },
    2: {
        "patch": "A",
        "color": "red",
        "reference_name": "A7",
        "reference_latlon": patch_a_microphones["A7"],
        "times": ["12:08:38", "12:09:08", "12:09:38", "12:10:08", "12:10:38"],
        "source_files": ["*_190000_SYNC.WAV"] * 5,
        "source_locs": patch_a_test2_all_playbacks_source_locs,
    },
    3: {
        "patch": "E",
        "color": "blue",
        "point_colors": ["blue"] * 5 + ["blue"] * 5,
        "reference_name": "E8",
        "reference_latlon": patch_e_microphones["E8"],
        "times": [
            "13:37:44.2", "13:38:13.8", "13:38:43.2", "13:39:13.6",
            "13:39:43.8", "13:40:13.8", "13:40:43.6", "13:41:13.8",
            "13:41:43.8", "13:42:13.8",
        ],
        "source_files": ["*_203000_SYNC.WAV"] * 10,
        "source_locs": patch_e_test3_all_playbacks_source_locs,
    },
    4: {
        "patch": "E",
        "color": "green",
        "point_colors": ["green"] + ["green"] * 5,
        "reference_name": "E2",
        "reference_latlon": patch_e_microphones["E2"],
        "times": [
            "13:57:54.4", "14:00:24.2", "14:00:54.2", "14:01:24.2",
            "14:01:54.2", "14:02:24.2",
        ],
        "source_files": [
            "*_203000_SYNC.WAV",
            "*_210000_SYNC.WAV", "*_210000_SYNC.WAV", "*_210000_SYNC.WAV",
            "*_210000_SYNC.WAV", "*_210000_SYNC.WAV",
        ],
        "source_locs": patch_e_test4_all_playbacks_source_locs,
    },
}

{test_num: len(test["source_locs"]) for test_num, test in playback_tests.items()}

{1: 5, 2: 5, 3: 10, 4: 6}

## Convert local east/north offsets to latitude/longitude

In [40]:
def xy_m_to_latlon(ref_lat, ref_lon, x_east_m, y_north_m):
    """Convert local east/north offsets in meters to [latitude, longitude]."""
    earth_radius_m = 6_371_000
    lat = ref_lat + np.degrees(y_north_m / earth_radius_m)
    mean_lat = (ref_lat + lat) / 2
    lon = ref_lon + np.degrees(
        x_east_m / (earth_radius_m * np.cos(np.radians(mean_lat)))
    )
    return [lat, lon]


for test in playback_tests.values():
    ref_lat, ref_lon = test["reference_latlon"]
    test["latlons"] = [
        xy_m_to_latlon(ref_lat, ref_lon, source[0], source[1])
        for source in test["source_locs"]
    ]

## Combined interactive map

In [46]:
map_center = [47.65495, -122.29575]

combined_map = folium.Map(
    location=map_center,
    zoom_start=18,
    control_scale=True,
    max_zoom=24,
    tiles=None,
)

folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors",
    name="OpenStreetMap",
    max_native_zoom=19,
    max_zoom=24,
).add_to(combined_map)

# Add every unique microphone from both patches as a black X.
microphone_group = folium.FeatureGroup(name="Microphones", show=True)
for name, location in microphone_locations.items():
    x_icon = folium.DivIcon(
        icon_size=(16, 16),
        icon_anchor=(8, 8),
        html=(
            '<div style="font-size:24px; font-weight:bold; color:black; '
            'line-height:16px; text-align:center;">X</div>'
        ),
    )
    folium.Marker(
        location=location,
        icon=x_icon,
        tooltip=folium.Tooltip(name, direction="right", offset=(8, 0)),
    ).add_to(microphone_group)

    folium.Marker(
            location=location,
            icon=folium.DivIcon(
                icon_size=(40, 20),
                icon_anchor=(-10, 8),  # Positions label to the right
                class_name="",
                html=(
                    '<div style="font-size:24px; font-weight:bold; '
                    'color:black; white-space:nowrap; '
                    f'text-shadow:1px 1px 2px white;">{name.split()[-1]}</div>'
                ),
            ),
        ).add_to(microphone_group)
microphone_group.add_to(combined_map)

# Add each playback test as a separate layer.
for test_num, test in playback_tests.items():
    test_group = folium.FeatureGroup(
        name=f'Playback test #{test_num} localization',
        show=True,
    )
    for point_num, (location, source, time) in enumerate(
        zip(test["latlons"], test["source_locs"], test["times"]),
        start=1,
    ):
        x_east, y_north = source[:2]
        source_files = test.get("source_files")
        source_file_html = (
            f'Source files: {source_files[point_num - 1]}<br>'
            if source_files is not None
            else ''
        )
        point_colors = test.get(
            "point_colors",
            [test["color"]] * len(test["source_locs"]),
        )
        point_color = point_colors[point_num - 1]
        tooltip = (
            f'Playback test #{test_num}, point {point_num}<br>'
            f'Time: {time}<br>'
            f'{source_file_html}'
            f'Patch: {test["patch"]}<br>'
            f'Reference microphone: {test["reference_name"]}<br>'
            f'East: {x_east:.2f} m<br>'
            f'North: {y_north:.2f} m'
        )
        folium.CircleMarker(
            location=location,
            radius=5,
            color="black",
            weight=1,
            fill=True,
            fill_color=point_color,
            fill_opacity=1,
            tooltip=folium.Tooltip(tooltip, sticky=True),
        ).add_to(test_group)
    test_group.add_to(combined_map)

MeasureControl(
    position="topleft",
    primary_length_unit="meters",
    secondary_length_unit="feet",
).add_to(combined_map)

folium.LayerControl(position="topright", collapsed=True).add_to(combined_map)

In [48]:
legend_rows = [
    ("red", "Playback test #1 localization"),
    ("blue", "Playback test #2 localization"),
    ("green", "Playback test #3 localization")
]

legend_items_html = "".join(
    f'''
    <div style="display:flex; align-items:center; margin-top:6px;">
        <span style="display:inline-block; width:10px; height:10px;
                     margin-left:4px; margin-right:12px;
                     background-color:{color}; border:1px solid black;
                     border-radius:50%;"></span>
        <span>{label}</span>
    </div>
    '''
    for color, label in legend_rows
)

legend_html = f'''
<div style="position:fixed; bottom:60px; left:40px; z-index:9999;
            background-color:white; border:2px solid #777; border-radius:5px;
            padding:10px 14px; font-size:18px;
            box-shadow:0 1px 5px rgba(0,0,0,0.35);">
    <div style="font-weight:bold; margin-bottom:8px;">Legend</div>
    <div style="display:flex; align-items:center;">
        <span style="display:inline-block; width:18px; margin-right:8px;
                     color:black; font-size:18px; font-weight:bold;
                     text-align:center;">X</span>
        <span>Microphone</span>
    </div>
    {legend_items_html}
</div>
'''

# combined_map.get_root().html.add_child(folium.Element(legend_html))

# Fit the initial view to all microphones and localizations.
all_locations = list(microphone_locations.values())
for test in playback_tests.values():
    all_locations.extend(test["latlons"])
combined_map.fit_bounds(all_locations, padding=(30, 30))

scale_bar_css = """
<style>
.leaflet-control-scale-line {
    font-size: 24px !important;
}
</style>
"""

combined_map.get_root().header.add_child(
    folium.Element(scale_bar_css)
)
combined_map

In [43]:
# Uncomment to save a standalone interactive HTML file.
combined_map.save("20260723__all_playback_test_localizations.html")

## Optional: save the interactive map